# Инициализация

Загружаем библиотеки необходимые для выполнения кода ноутбука.

In [1]:
import pandas as pd

# === ЭТАП 1 ===

# Загрузка первичных данных

Загружаем первичные данные из файлов:
- tracks.parquet
- catalog_names.parquet
- interactions.parquet

In [2]:
tracks = pd.read_parquet("tracks.parquet")
catalog_names = pd.read_parquet("catalog_names.parquet")
interactions = pd.read_parquet("interactions.parquet")

In [3]:
display(tracks.head())
display(catalog_names.sample(5).head())
display(interactions.sample(5).head())

,track_id,albums,artists,genres
0,26,"[3, 2490753]",[16],"[11, 21]"
1,38,"[3, 2490753]",[16],"[11, 21]"
2,135,"[12, 214, 2490809]",[84],[11]
3,136,"[12, 214, 2490809]",[84],[11]
4,138,"[12, 214, 322, 72275, 72292, 91199, 213505, 24...",[84],[11]


,id,type,name
990448,4855083,track,Sleeping
887420,669799,track,It's Alright (Baby's Coming Back)
494177,10884351,album,Нарушители пустоты
438515,8857566,album,think of me
953324,2830410,track,Дым и вода


,user_id,track_id,track_seq,started_at
957,1285658,41418309,958,2022-10-13
188,968998,37138319,189,2022-09-25
48,977525,1531474,49,2022-12-02
798,1097982,37606534,799,2022-10-23
15,1286610,63196880,16,2022-10-22


# Обзор данных

Проверяем данные, есть ли с ними явные проблемы.

In [4]:
# Анализ идентификаторов треков
print(tracks['track_id'].dtypes)
print(tracks['track_id'].duplicated().sum()) # Не имеет дубликатов track_id
print(tracks['track_id'].isnull().sum())

int64
0
0


In [5]:
album_ids = tracks['albums'].explode()
artist_ids = tracks['artists'].explode()
genre_id = tracks['genres'].explode()

# Проверка есть ли треки с неизвестными исполнителями, альбомами, жанрами
print(album_ids.isnull().sum())
print(artist_ids.isnull().sum())
print(genre_id.isnull().sum())
# Такие треки есть

#Удаляем треки с пропусками
indices_to_drop = (album_ids[album_ids.isnull()].index
                   .union(artist_ids[artist_ids.isnull()].index)
                   .union(genre_id[genre_id.isnull()].index))
tracks = tracks.drop(index=indices_to_drop)
tracks.reset_index(drop=True, inplace=True)

# Очередная проверка есть ли треки с неизвестными исполнителями, альбомами, жанрами
album_ids = tracks['albums'].explode()
artist_ids = tracks['artists'].explode()
genre_id = tracks['genres'].explode()
print(album_ids.isnull().sum())
print(artist_ids.isnull().sum())
print(genre_id.isnull().sum())

18
15369
3687
0
0
0


In [6]:
#Проверка на пропуски
print(catalog_names.isnull().sum(axis=1).sum(), 'Пропусков в catalog_names') # Пропусков нет

# Проверка на пересечение идентификаторов
print(album_ids.isin(artist_ids).sum(), album_ids.isin(genre_id).sum(), artist_ids.isin(genre_id).sum(), 'Пересечение id у album, artist, genre') # Идентификаторы пересекаются, но у них разный type, объект можно идентифицировать по id + type

#Проверка на дупликаты
print(catalog_names.duplicated().sum(), 'Кол-во дубликтов в catalog_names') # Дублей нету

def check_object_ids(catalog_names: pd.DataFrame, type: str, ids: pd.Series):
    uniq_ids = ids.unique()
    catalog_type_ids = catalog_names.loc[catalog_names.type == type]['id'].unique()
    print(f'Для типа {type} у треков {len(uniq_ids)} уникальных id; У catalog_names {len(catalog_type_ids)}; {len(list(set(uniq_ids) - set(catalog_type_ids)))} есть id, которых по типу {type} не у каталогов')
    return list(set(uniq_ids) - set(catalog_type_ids))
    
#Проверка наличия идентификаторов в catalog_names
print((~pd.Series(album_ids.unique()).isin(catalog_names.loc[catalog_names['type'] == 'album']['id'])).sum(), 'Кол-во не хватающих id альбомов в catalog_names') # Все id альбомов есть в catalog_names
print((~pd.Series(artist_ids.unique()).isin(catalog_names.loc[catalog_names['type'] == 'artist']['id'])).sum(), 'Кол-во не хватающих id артистов в catalog_names') # Все id артистов есть в catalog_names
print((~pd.Series(genre_id.unique()).isin(catalog_names.loc[catalog_names['type'] == 'genre']['id'])).sum(), 'Кол-во не хватающих id жанров в catalog_names') # id жанров не хватает в catalog_names
print((~tracks['track_id'].isin(catalog_names.loc[catalog_names['type'] == 'track']['id'])).sum(), 'Кол-во не хватающих id треков в catalog_names') # Все id треков есть в catalog_names
check_object_ids(catalog_names, 'album', album_ids)
check_object_ids(catalog_names, 'artist', artist_ids)
no_genre_ids = check_object_ids(catalog_names, 'genre', genre_id)
print(no_genre_ids)
display(catalog_names.loc[catalog_names.id.isin(no_genre_ids) & (catalog_names.type != 'genre')])


0 Пропусков в catalog_names
64183 370 1297 Пересечение id у album, artist, genre
0 Кол-во дубликтов в catalog_names
0 Кол-во не хватающих id альбомов в catalog_names
0 Кол-во не хватающих id артистов в catalog_names
30 Кол-во не хватающих id жанров в catalog_names
0 Кол-во не хватающих id треков в catalog_names
Для типа album у треков 653563 уникальных id; У catalog_names 658724; 0 есть id, которых по типу album не у каталогов
Для типа artist у треков 152785 уникальных id; У catalog_names 153581; 0 есть id, которых по типу artist не у каталогов
Для типа genre у треков 166 уникальных id; У catalog_names 166; 30 есть id, которых по типу genre не у каталогов
[130, 131, 132, 133, 134, 135, 146, 148, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 124, 126]


,id,type,name
49,126,album,Arma-goddamn-motherfuckin-geddon
51,132,album,LoveGame Remixes
52,133,album,Best Of
53,134,album,Specialty Profiles: Roy Milton
54,135,album,LoveGame
61,146,album,Zee Avi
62,148,album,In A Perfect World...
64,150,album,B Is For Bob
65,152,album,Trojan Lovers Collection
66,154,album,Stop!


In [7]:
mask = tracks['genres'].apply(lambda x: not set(x).isdisjoint(no_genre_ids))
tracks[mask]
#48313 треков, у которых в жанре есть id, которого нету в справочнике catalog_names


,track_id,albums,artists,genres
36,436,[36],[330],"[28, 164]"
59,594,"[54, 88, 5479, 5785124, 9198099, 9231427, 1088...",[533],"[28, 162]"
125,1025,"[94, 2325, 8757, 8986, 318695, 17004129]",[937],"[28, 162]"
126,1026,"[94, 780, 8727]",[937],"[28, 162]"
128,1028,"[94, 4865, 19666753, 19666891]",[936],"[28, 162]"
...,...,...,...,...
980753,101009893,[21261051],"[9772048, 9772047]","[47, 161]"
980781,101049628,[21272435],[5926594],"[47, 161]"
980845,101200283,[21314376],[4130480],"[47, 154]"
980886,101256806,[21331759],[5926594],"[47, 161]"


In [8]:
#Проверяем типы данных идентификаторов
display(tracks.dtypes)
display(catalog_names.dtypes)
display(interactions.dtypes)
#Можно заметить тип данных в intersactions у track_id int32, а у tracks int64 => можно скастить типы для экономии памяти и избежания переполнения int32 в случае большего значения id в tracks

track_id     int64
albums      object
artists     object
genres      object
dtype: object

id       int64
type    object
name    object
dtype: object

user_id                int32
track_id               int32
track_seq              int16
started_at    datetime64[ns]
dtype: object

In [9]:
#Проверяем дубликаты и пропущенные значения
print(catalog_names.duplicated().sum(), 'Кол-во дубликатов у catalog_names', catalog_names.isnull().sum(axis=1).sum(), 'Кол-во пропущенных значений')
print(interactions.duplicated().sum(), 'Кол-во дубликатов у interactions', catalog_names.isnull().sum(axis=1).sum(), 'Кол-во пропущенных значений')


0 Кол-во дубликатов у catalog_names 0 Кол-во пропущенных значений
0 Кол-во дубликатов у interactions 0 Кол-во пропущенных значений


In [10]:
# Испрвляем оставшиеся проблемы
# Удаляем треки, у которых не сущ в catalog_names жанры есть
tracks = tracks[~mask]
del mask # Освобождаем память


In [11]:
# Меняем тип данных track_id в tracks
tracks['track_id'] = tracks['track_id'].astype('int32')

In [13]:
del indices_to_drop

In [14]:
del check_object_ids

In [12]:
# Удаляем из interactions треки, которых нет в tracks
interactions = interactions.loc[(interactions.track_id.isin(tracks.track_id))]
del no_genre_ids, album_ids, artist_ids, genre_id


# Выводы

Приведём выводы по первому знакомству с данными:
- есть ли с данными явные проблемы,
- какие корректирующие действия (в целом) были предприняты.

## В ходе подготовки данных были выявлены след. проблемы: 
- типы данных для track_id у interactions и tracks различаются;
- есть треки с пропусками данных в колонках: albums, artists, genres
- есть треки, которые в genres имеют идентификаторы, которых нету в справочнике catalog_names с type = 'genre'
- в следствии проблем выше с треками в interactions есть записи, которые нужно удалить после очистки треков.

## Предпринятые шаги решения проблем:
- каст типов track_id
- удаление треков с пропусками в колонках albums, artists, genres
- удаление треков, содержащих genres, которых нет в catalog_names
- очистка interactions

# === ЭТАП 2 ===

# EDA

Распределение количества прослушанных треков.

: 

: 

Наиболее популярные треки

: 

: 

Наиболее популярные жанры

: 

: 

Треки, которые никто не прослушал

: 

: 

# Преобразование данных

Преобразуем данные в формат, более пригодный для дальнейшего использования в расчётах рекомендаций.

: 

: 

# Сохранение данных

Сохраним данные в двух файлах в персональном S3-бакете по пути `recsys/data/`:
- `items.parquet` — все данные о музыкальных треках,
- `events.parquet` — все данные о взаимодействиях.

: 

: 

# Очистка памяти

Здесь, может понадобится очистка памяти для высвобождения ресурсов для выполнения кода ниже. 

Приведите соответствующие код, комментарии, например:
- код для удаление более ненужных переменных,
- комментарий, что следует перезапустить kernel, выполнить такие-то начальные секции и продолжить с этапа 3.

: 

: 

# === ЭТАП 3 ===

# Загрузка данных

Если необходимо, то загружаем items.parquet, events.parquet.

: 

: 

: 

: 

# Разбиение данных

Разбиваем данные на тренировочную, тестовую выборки.

: 

: 

: 

: 

# Топ популярных

Рассчитаем рекомендации как топ популярных.

: 

: 

: 

: 

# Персональные

Рассчитаем персональные рекомендации.

: 

: 

: 

: 

# Похожие

Рассчитаем похожие, они позже пригодятся для онлайн-рекомендаций.

: 

: 

: 

: 

# Построение признаков

Построим три признака, можно больше, для ранжирующей модели.

: 

: 

: 

: 

# Ранжирование рекомендаций

Построим ранжирующую модель, чтобы сделать рекомендации более точными. Отранжируем рекомендации.

: 

: 

: 

: 

# Оценка качества

Проверим оценку качества трёх типов рекомендаций: 

- топ популярных,
- персональных, полученных при помощи ALS,
- итоговых
  
по четырем метрикам: recall, precision, coverage, novelty.

: 

: 

: 

: 

# === Выводы, метрики ===

Основные выводы при работе над расчётом рекомендаций, рассчитанные метрики.

: 

: 

: 

: 